<a href="https://colab.research.google.com/github/edik06031-rgb/DTA_2026/blob/main/ML/ML_feature_engineering_categorical_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engineering. Categorical features: One-Hot, Ordinal

## Налаштування та дані

- квартири (регресія) — з числовими та категорійними ознаками й датою

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# ---------- Датасет 1: КВАРТИРИ (регресія) ----------
n = 1500
cities = np.random.choice(["Київ", "Львів", "Харків", "Одеса"], n, p=[.4, .2, .2, .2])
city_premium = pd.Series({"Київ": 60, "Львів": 25, "Харків": 10, "Одеса": 20})

condition = np.random.choice(["аварійний", "житловий", "хороший", "євроремонт"],
                             n, p=[.1, .4, .35, .15])
cond_bonus = pd.Series({"аварійний": -20, "житловий": 0, "хороший": 15, "євроремонт": 40})

area    = np.random.normal(60, 20, n).clip(20, 140)
rooms   = np.clip(np.round(area / 25 + np.random.normal(0, .6, n)), 1, 5).astype(int)
floor   = np.random.randint(1, 25, n)
dist_km = np.random.exponential(5, n).clip(.3, 25)
listing_date = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 540, n), unit="D")

price = (40 + area*1.8 + rooms*5 + floor*.4 - dist_km*3
         + city_premium[cities].values + cond_bonus[condition].values
         + np.random.normal(0, 12, n)).clip(20, None)

apt = pd.DataFrame({
    "area": area.round(1), "rooms": rooms, "floor": floor,
    "dist_km": dist_km.round(1), "city": cities, "condition": condition,
    "listing_date": listing_date, "price": price.round(1),
})


print("Квартири:", apt.shape)
apt.head()

Квартири: (1500, 8)


,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


## Feature Engineering — створення нових ознак

In [3]:
df = apt.copy()
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


In [4]:
df["area_per_room"] = (df.area / df.rooms).round(1)               # співвідношення
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price,area_per_room
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7,27.1
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3,24.1
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1,18.4
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1,32.7
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1,28.1


In [8]:
# взаємодія: пошук великої квартири яка знаходиться далеко від центру
df["area_x_dist"] = (df.area * df.dist_km).round(1)
df[["area","dist_km","area_x_dist"]].head()
df[["area","dist_km","area_x_dist"]].describe().round(1)

,area,dist_km,area_x_dist
count,1500.0,1500.0,1500.0
mean,60.0,4.7,281.3
std,19.9,4.5,298.7
min,20.0,0.3,6.0
25%,45.6,1.5,74.2
50%,59.8,3.2,182.0
75%,73.4,6.4,377.2
max,138.5,25.0,1880.0


In [10]:
# Бінарна ознака : True(1) & False(0)
df["is_central"] = (df.dist_km < 3).astype(int)
df[["dist_km", "is_central"]].head()

,dist_km,is_central
0,4.4,0
1,4.0,0
2,1.2,1
3,2.8,1
4,5.3,0


In [13]:
# бін - поділ на групи
df["floor_group"] = pd.cut(
    df.floor,
    bins=[0, 2, 9, 100],
    labels = ["löw", "median", "high"]
)
df[["floor_group", "floor"]].head()

,floor_group,floor
0,high,20
1,high,10
2,high,23
3,median,9
4,löw,2
